# Análises de Negócio

Análises construídas sobre a camada Gold, organizadas pelas dores relatadas por cada parte interessada da V-Credit: o Gerente Financeiro, a Coordenadora de Customer Success, o Diretor de Tecnologia, a Diretora de Marketing e os atendentes. Cada seção reproduz o relato e reúne as consultas que o testam.

Convenções usadas nas consultas: em `FK_status_chamado`, 1 é resolvido, 2 é não resolvido (o cliente foi atendido) e 3 é abandonado (o cliente desistiu na espera). Chaves iguais a -1 apontam para o membro "Não Informado" das dimensões e são excluídas quando a análise depende daquela dimensão. O CSAT é medido como o percentual de notas 4 ou 5 entre os chamados avaliados.

In [0]:
%sql
USE CATALOG vcredit_medallion;

## Visão geral

Indicadores totais da operação: volume de chamados, custo total e médio, distribuição por status, cobertura e resultado da pesquisa de satisfação e tempos médios de espera e de atendimento.

In [0]:
%sql
SELECT
  COUNT(*)                                                    AS total_chamados,
  ROUND(SUM(custo), 2)                                        AS custo_total,
  ROUND(AVG(custo), 2)                                        AS custo_medio_por_chamado,
  ROUND(100 * COUNT_IF(FK_status_chamado = 1) / COUNT(*), 1)  AS pct_resolvidos,
  ROUND(100 * COUNT_IF(FK_status_chamado = 2) / COUNT(*), 1)  AS pct_nao_resolvidos,
  ROUND(100 * COUNT_IF(FK_status_chamado = 3) / COUNT(*), 1)  AS pct_abandonados,
  ROUND(100 * COUNT_IF(nota_avaliacao IS NOT NULL) / COUNT(*), 1) AS pct_com_avaliacao,
  ROUND(AVG(nota_avaliacao), 2)                               AS nota_media,
  ROUND(100 * COUNT_IF(nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5,
  ROUND(AVG(tempo_espera), 1)                                 AS tempo_medio_espera_seg,
  ROUND(AVG(tempo_atendimento), 1)                            AS tempo_medio_atendimento_seg
FROM gold.fato_chamado;

## 1. Gerente Financeiro: contenção de custo

> "O gasto com atendimento passou de R$ 1,4 milhão nos últimos dezenove meses e cresce todo mês. O crescimento não acompanha a receita. Precisamos cortar essa despesa."

As consultas desta seção mostram como o custo evolui no tempo, onde ele se concentra (canal, motivo e nível do atendente) e quanto é gasto em chamados que terminam sem resolução.

### 1.1 Evolução mensal do custo

Custo total e médio por mês, separando o que foi gasto em chamados resolvidos e não resolvidos.

In [0]:
%sql
-- Mostra se o custo cresce mês a mês; separa custo de resolvidos vs não resolvidos
SELECT
  d.ano,
  d.mes,
  COUNT(*)                         AS chamados,
  ROUND(SUM(f.custo), 2)           AS custo_total,
  ROUND(AVG(f.custo), 2)           AS custo_medio,
  ROUND(SUM(CASE WHEN f.FK_status_chamado = 1 THEN f.custo ELSE 0 END), 2)  AS custo_resolvidos,
  ROUND(SUM(CASE WHEN f.FK_status_chamado <> 1 THEN f.custo ELSE 0 END), 2) AS custo_nao_resolvidos
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

### 1.2 Custo por canal

Custo total, custo médio e participação de cada canal no custo total.

In [0]:
%sql
-- Onde o dinheiro está concentrado?
SELECT
  c.nome                        AS canal,
  c.tipo                        AS tipo_canal,
  COUNT(*)                      AS chamados,
  ROUND(SUM(f.custo), 2)        AS custo_total,
  ROUND(AVG(f.custo), 2)        AS custo_medio,
  ROUND(100 * SUM(f.custo) / (SELECT SUM(custo) FROM gold.fato_chamado), 1) AS pct_custo_total
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.SK_canal <> -1
GROUP BY c.nome, c.tipo
ORDER BY custo_total DESC;

### 1.3 Custo por motivo

Os 10 motivos de maior custo total, com custo médio e taxa de resolução.

In [0]:
%sql
-- Quais problemas custam mais para a empresa?
SELECT
  m.nome                        AS motivo,
  m.categoria,
  m.criticidade,
  COUNT(*)                      AS chamados,
  ROUND(SUM(f.custo), 2)        AS custo_total,
  ROUND(AVG(f.custo), 2)        AS custo_medio,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao
FROM gold.fato_chamado f
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
WHERE m.SK_motivo <> -1
GROUP BY m.nome, m.categoria, m.criticidade
ORDER BY custo_total DESC
LIMIT 10;

### 1.4 Custo de ineficiência

Quanto do custo total corresponde a chamados resolvidos, não resolvidos e abandonados.

In [0]:
%sql
-- Quanto dinheiro é gasto em chamados que não geram valor?
SELECT
  CASE
    WHEN f.FK_status_chamado = 1 THEN 'Resolvido'
    WHEN f.FK_status_chamado = 2 THEN 'Não resolvido (atendido)'
    WHEN f.FK_status_chamado = 3 THEN 'Abandonado (na espera)'
  END                          AS status_chamado,
  COUNT(*)                     AS chamados,
  ROUND(SUM(f.custo), 2)       AS custo_total,
  ROUND(100 * SUM(f.custo) / (SELECT SUM(custo) FROM gold.fato_chamado), 1) AS pct_custo_total
FROM gold.fato_chamado f
GROUP BY 1
ORDER BY custo_total DESC;

### 1.5 Custo e eficiência por nível de atendente

Compara o nível 1 (atendimento inicial) e o nível 2 (atendimento especializado) em custo, taxa de resolução e tempo médio de atendimento.

In [0]:
%sql
-- Nível 1 = Atendimento Inicial; Nível 2 = Atendimento Especializado
SELECT
  a.nivel_atendimento,
  COUNT(*)                              AS chamados,
  ROUND(SUM(f.custo), 2)               AS custo_total,
  ROUND(AVG(f.custo), 2)               AS custo_medio,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(AVG(f.tempo_atendimento), 1)    AS tempo_medio_atend_seg
FROM gold.fato_chamado f
JOIN gold.dim_atendente a ON f.FK_atendente = a.SK_atendente
WHERE a.SK_atendente <> -1
GROUP BY a.nivel_atendimento
ORDER BY a.nivel_atendimento;

## 2. Coordenadora de Customer Success: qualidade do atendimento

> "O tempo médio de atendimento subiu e a taxa de resolução caiu, principalmente no nível 2. Muitos clientes desistem antes de serem atendidos."

As consultas desta seção verificam a evolução do tempo e da resolução, o abandono por canal e por tempo de espera, o desempenho individual dos atendentes e a satisfação por motivo e canal.

### 2.1 Evolução mensal do tempo e da resolução

Tempos médios de espera e de atendimento, taxa de resolução e taxa de abandono por mês.

In [0]:
%sql
-- Confirma se o tempo sobe e a resolução cai ao longo do tempo
SELECT
  d.ano,
  d.mes,
  COUNT(*)                                                  AS chamados,
  ROUND(AVG(f.tempo_atendimento), 1)                         AS tempo_medio_atend_seg,
  ROUND(AVG(f.tempo_espera), 1)                             AS tempo_medio_espera_seg,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

### 2.2 Resolução e abandono por canal

Taxas de resolução, não resolução e abandono e tempos médios de cada canal, na ordem das etapas do funil.

In [0]:
%sql
-- Onde a resolução é mais crítica (especialmente nível 2 / Atendimento Especializado)?
SELECT
  c.nome                       AS canal,
  c.tipo                       AS tipo_canal,
  c.nivel_etapa,
  COUNT(*)                     AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 2) / COUNT(*), 1) AS pct_nao_resolvido,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono,
  ROUND(AVG(f.tempo_atendimento), 1)  AS tempo_medio_atend_seg,
  ROUND(AVG(f.tempo_espera), 1)       AS tempo_medio_espera_seg
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.SK_canal <> -1
GROUP BY c.nome, c.tipo, c.nivel_etapa
ORDER BY c.nivel_etapa;

### 2.3 Abandono por tempo de espera

Nos canais humanos, relaciona faixas de tempo de espera com as taxas de abandono e de resolução.

In [0]:
%sql
-- Clientes desistem antes de serem atendidos: qual é o impacto do tempo de espera?
SELECT
  CASE
    WHEN f.tempo_espera = 0 THEN '0 min (sem espera)'
    WHEN f.tempo_espera <= 60 THEN 'até 1 min'
    WHEN f.tempo_espera <= 180 THEN '1 a 3 min'
    WHEN f.tempo_espera <= 300 THEN '3 a 5 min'
    WHEN f.tempo_espera <= 600 THEN '5 a 10 min'
    ELSE 'mais de 10 min'
  END                        AS faixa_espera,
  COUNT(*)                   AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.tipo = 'Humano'
GROUP BY 1
ORDER BY MIN(f.tempo_espera);

### 2.4 Ranking de atendentes

Volume, tempo médio, taxa de resolução e satisfação de cada atendente, para orientar capacitação.

In [0]:
%sql
-- Quem performa melhor/pior para orientar capacitação
SELECT
  a.nome,
  a.nivel_atendimento,
  COUNT(*)                                                  AS chamados,
  ROUND(AVG(f.tempo_atendimento), 1)                       AS tempo_medio_atend_seg,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(AVG(f.nota_avaliacao), 2)                           AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5,
  COUNT_IF(f.nota_avaliacao IS NOT NULL)                    AS avaliacoes
FROM gold.fato_chamado f
JOIN gold.dim_atendente a ON f.FK_atendente = a.SK_atendente
WHERE a.SK_atendente <> -1
GROUP BY a.nome, a.nivel_atendimento
ORDER BY pct_resolucao DESC, tempo_medio_atend_seg ASC;

### 2.5 Satisfação por motivo e canal

As 15 combinações de motivo e canal com menor CSAT, considerando apenas as que têm mais de 10 avaliações.

In [0]:
%sql
-- Onde a satisfação é menor e precisa de atenção
SELECT
  m.categoria,
  m.nome                        AS motivo,
  c.nome                        AS canal,
  COUNT(*)                      AS chamados,
  COUNT_IF(f.nota_avaliacao IS NOT NULL) AS com_avaliacao,
  ROUND(AVG(f.nota_avaliacao), 2) AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5
FROM gold.fato_chamado f
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE m.SK_motivo <> -1 AND c.SK_canal <> -1
GROUP BY m.categoria, m.nome, c.nome
HAVING COUNT_IF(f.nota_avaliacao IS NOT NULL) > 10
ORDER BY csat_pct_4_5 ASC
LIMIT 15;

## 3. Diretor de Tecnologia: chatbot e integração de dados

> "O chatbot não resolve a maior parte das solicitações. Há inconsistências entre os registros de atendimento, custos e notas de satisfação. Precisamos estruturar melhor a integração e a segurança."

As consultas desta seção comparam o chatbot com os demais canais, identificam os motivos que ele resolve pior, testam a hipótese de que o perfil etário explica a baixa adesão e medem a cobertura dos dados entre os sistemas.

### 3.1 Efetividade do chatbot frente aos outros canais

Resolução, abandono, tempo médio e custo total de cada canal.

In [0]:
%sql
-- O chatbot resolve a maioria das solicitações? Como compara com URA e humanos?
SELECT
  c.nome                        AS canal,
  c.tipo                        AS tipo_canal,
  COUNT(*)                      AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono,
  ROUND(AVG(f.tempo_atendimento), 1)  AS tempo_medio_atend_seg,
  ROUND(SUM(f.custo), 2)              AS custo_total
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.SK_canal <> -1
GROUP BY c.nome, c.tipo, c.nivel_etapa
ORDER BY c.nivel_etapa;

### 3.2 Resolução do chatbot por motivo

Taxa de resolução do chatbot para cada motivo, da menor para a maior.

In [0]:
%sql
-- Quais tipos de solicitação o chatbot consegue resolver e quais não?
SELECT
  m.nome                        AS motivo,
  m.categoria,
  COUNT(*)                      AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao_chatbot
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
WHERE c.nome = 'Chatbot' AND m.SK_motivo <> -1
GROUP BY m.nome, m.categoria
ORDER BY pct_resolucao_chatbot ASC;

### 3.3 Perfil etário por canal

Distribuição dos chamados de cada canal por faixa de idade do cliente. Testa a afirmação de que clientes com pouca familiaridade digital não conseguem usar o chatbot.

In [0]:
%sql
-- O Diretor de TI levanta que "analfabetos digitais" não usam bem o chatbot
-- A idade dos clientes do chatbot difere dos que usam atendimento humano?
SELECT
  c.nome                        AS canal,
  cl.faixa_idade,
  COUNT(*)                       AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY c.nome), 1) AS pct_no_canal
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
JOIN gold.dim_cliente cl ON f.FK_cliente = cl.SK_cliente
WHERE c.SK_canal <> -1 AND cl.SK_cliente <> -1 AND cl.faixa_idade <> 'Não Informado'
GROUP BY c.nome, cl.faixa_idade
ORDER BY c.nome,
  CASE cl.faixa_idade
    WHEN '18-24' THEN 1 WHEN '25-34' THEN 2 WHEN '35-44' THEN 3
    WHEN '45-54' THEN 4 WHEN '55-64' THEN 5 WHEN '65-74' THEN 6
    WHEN '75-84' THEN 7 WHEN '85+' THEN 8
  END;

### 3.4 Cobertura dos dados entre sistemas

Percentual de chamados sem custo, sem nota de satisfação e sem cliente, atendente ou data de início identificados. Dimensiona as inconsistências entre os sistemas de atendimento, custos e satisfação.

In [0]:
%sql
-- % de chamados sem custo registrado, sem nota de satisfação, e FKs apontando para "Não Informado"
-- Quantifica o tamanho das inconsistências entre os sistemas de atendimento, custos e satisfação
SELECT
  ROUND(100 * COUNT_IF(custo = 0) / COUNT(*), 1)                    AS pct_sem_custo,
  ROUND(100 * COUNT_IF(nota_avaliacao IS NULL) / COUNT(*), 1)       AS pct_sem_nota,
  ROUND(100 * COUNT_IF(FK_cliente = -1) / COUNT(*), 1)              AS pct_cliente_nao_identificado,
  ROUND(100 * COUNT_IF(FK_atendente = -1) / COUNT(*), 1)           AS pct_atendente_nao_identificado,
  ROUND(100 * COUNT_IF(FK_data_inicio_atendimento = -1) / COUNT(*), 1) AS pct_sem_data_inicio,
  COUNT(*)                                                          AS total_chamados
FROM gold.fato_chamado;

## 4. Diretora de Marketing: satisfação do cliente

> "No começo de 2025 nossos clientes nos amavam, o coeficiente de satisfação da V-Credit era 85%. Sentimos uma piora recente, mas acredito que esteja ligada a fatores externos e não à nossa operação. Não deve estar muito diferente do que vimos no ano passado."

O relato traz quatro afirmações que podem ser testadas com os dados: a satisfação era de cerca de 85% no início de 2025, houve piora recente, a piora tem causa externa e o resultado atual é parecido com o do ano anterior.

Um fator externo, como economia, concorrência ou sazonalidade, tenderia a derrubar a nota de forma uniforme em todos os canais, motivos e níveis, e também no mesmo período do ano anterior. Se a queda se concentra em canais ou níveis específicos, ou acompanha o aumento de espera, abandono e não resolução, a causa é operacional. Como os dados vão até julho de 2026, as comparações entre anos usam apenas os meses de janeiro a julho.

### 4.1 Evolução mensal da satisfação

CSAT mensal em duas definições: o percentual de notas 4 ou 5 e a nota média convertida para a escala de 0 a 100.

In [0]:
%sql
-- csat_pct_4_5: % de notas 4 ou 5 entre os chamados avaliados
-- csat_nota_media_pct: nota média / 5 (escala 0–100)
SELECT
  d.ano,
  d.mes,
  COUNT_IF(f.nota_avaliacao IS NOT NULL)                         AS avaliacoes,
  ROUND(AVG(f.nota_avaliacao), 2)                                AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5,
  ROUND(100 * AVG(f.nota_avaliacao) / 5, 1)                      AS csat_nota_media_pct
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

### 4.2 Comparação entre 2025 e 2026

CSAT de cada mês de janeiro a julho nos dois anos, com a variação em pontos percentuais. Testa a afirmação de que o resultado não é diferente do ano anterior.

In [0]:
%sql
-- Dados vão até jul/2026, então só jan–jul permite comparar nas mesmas condições
SELECT
  d.mes,
  ROUND(100 * COUNT_IF(d.ano = 2025 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2025 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(d.ano = 2026 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2026 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026,
  ROUND(
      100 * COUNT_IF(d.ano = 2026 AND f.nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(d.ano = 2026 AND f.nota_avaliacao IS NOT NULL), 0)
    - 100 * COUNT_IF(d.ano = 2025 AND f.nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(d.ano = 2025 AND f.nota_avaliacao IS NOT NULL), 0), 1)    AS variacao_pp,
  COUNT_IF(d.ano = 2025)                                                              AS chamados_2025,
  COUNT_IF(d.ano = 2026)                                                              AS chamados_2026
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1 AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
GROUP BY d.mes
ORDER BY d.mes;

### 4.3 Satisfação por status do chamado

Nota média de chamados resolvidos, não resolvidos e abandonados em cada ano. Se a nota depende da resolução, a satisfação é consequência da operação.

In [0]:
%sql
-- Se a nota depende de o problema ter sido resolvido, a satisfação é função da operação
SELECT
  d.ano,
  CASE f.FK_status_chamado
    WHEN 1 THEN 'Resolvido'
    WHEN 2 THEN 'Não resolvido (atendido)'
    WHEN 3 THEN 'Abandonado (na espera)'
  END                                                              AS status_chamado,
  COUNT(*)                                                         AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.ano), 1) AS pct_do_ano,
  COUNT_IF(f.nota_avaliacao IS NOT NULL)                           AS avaliacoes,
  ROUND(AVG(f.nota_avaliacao), 2)                                  AS nota_media
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, f.FK_status_chamado
ORDER BY d.ano, f.FK_status_chamado;

### 4.4 Variação por canal

CSAT e participação no volume de cada canal em 2025 e 2026. Uma queda concentrada em poucos canais indica causa operacional.

In [0]:
%sql
-- Compara jan–jul/2025 x jan–jul/2026 por canal: CSAT, variação e mudança de peso no volume
WITH base AS (
  SELECT
    c.nome AS canal, c.tipo AS tipo_canal, c.nivel_etapa, d.ano,
    f.nota_avaliacao
  FROM gold.fato_chamado f
  JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
  WHERE d.SK_data <> -1 AND c.SK_canal <> -1
    AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
)
SELECT
  canal, tipo_canal, nivel_etapa,
  ROUND(100 * COUNT_IF(ano = 2025) / SUM(COUNT_IF(ano = 2025)) OVER (), 1) AS pct_volume_2025,
  ROUND(100 * COUNT_IF(ano = 2026) / SUM(COUNT_IF(ano = 2026)) OVER (), 1) AS pct_volume_2026,
  ROUND(100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026,
  ROUND(
      100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0)
    - 100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS variacao_pp
FROM base
GROUP BY canal, tipo_canal, nivel_etapa
ORDER BY variacao_pp ASC;

### 4.5 Variação por categoria e criticidade do motivo

CSAT em 2025 e 2026 por categoria e criticidade. Uma queda concentrada em categorias específicas aponta para falha de processo ou de produto.

In [0]:
%sql
-- Queda concentrada em categorias específicas aponta para falha de processo/produto, não para macroeconomia
WITH base AS (
  SELECT m.categoria, m.criticidade, d.ano, f.nota_avaliacao
  FROM gold.fato_chamado f
  JOIN gold.dim_data   d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
  WHERE d.SK_data <> -1 AND m.SK_motivo <> -1
    AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
)
SELECT
  categoria, criticidade,
  COUNT_IF(ano = 2026)                                                           AS chamados_2026,
  ROUND(100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026,
  ROUND(
      100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0)
    - 100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS variacao_pp
FROM base
GROUP BY categoria, criticidade
ORDER BY variacao_pp ASC;

### 4.6 Satisfação por tempo de espera

Nos canais humanos, CSAT e distribuição dos chamados por faixa de espera em cada ano. Mostra se a nota cai com a espera e se mais chamados passaram para as faixas longas em 2026.

In [0]:
%sql
-- Dois efeitos: (1) nota cai quando a espera aumenta; (2) mais chamados caem nas faixas longas em 2026
SELECT
  d.ano,
  CASE
    WHEN f.tempo_espera = 0   THEN '0 min (sem espera)'
    WHEN f.tempo_espera <= 60  THEN 'até 1 min'
    WHEN f.tempo_espera <= 180 THEN '1 a 3 min'
    WHEN f.tempo_espera <= 300 THEN '3 a 5 min'
    WHEN f.tempo_espera <= 600 THEN '5 a 10 min'
    ELSE 'mais de 10 min'
  END                                                              AS faixa_espera,
  COUNT(*)                                                         AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.ano), 1) AS pct_do_ano,
  ROUND(AVG(f.nota_avaliacao), 2)                                  AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5
FROM gold.fato_chamado f
JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE d.SK_data <> -1 AND c.tipo = 'Humano' AND d.ano IN (2025, 2026)
GROUP BY d.ano, 2
ORDER BY d.ano,
  CASE faixa_espera
    WHEN '0 min (sem espera)' THEN 1
    WHEN 'até 1 min'          THEN 2
    WHEN '1 a 3 min'          THEN 3
    WHEN '3 a 5 min'          THEN 4
    WHEN '5 a 10 min'         THEN 5
    ELSE 6
  END;

### 4.7 Viés de resposta da pesquisa

Taxa de resposta à pesquisa por status do chamado. Clientes que abandonam o atendimento raramente avaliam, então um aumento do abandono faz o CSAT medido parecer melhor do que a satisfação real.

In [0]:
%sql
-- Cliente que abandona normalmente não avalia. Se o abandono cresce, o CSAT medido
-- pode estar SUPERESTIMADO: a piora real seria maior do que a que o Marketing enxerga
SELECT
  d.ano,
  CASE f.FK_status_chamado
    WHEN 1 THEN 'Resolvido'
    WHEN 2 THEN 'Não resolvido (atendido)'
    WHEN 3 THEN 'Abandonado (na espera)'
  END                                                              AS status_chamado,
  COUNT(*)                                                         AS chamados,
  ROUND(100 * COUNT_IF(f.nota_avaliacao IS NOT NULL) / COUNT(*), 1) AS taxa_resposta_pct
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, f.FK_status_chamado
ORDER BY d.ano, f.FK_status_chamado;

### 4.8 Satisfação por perfil do cliente

CSAT em 2025 e 2026 por faixa etária e região. Uma queda parecida em todos os perfis indica problema no serviço, e não em um público específico.

In [0]:
%sql
-- Se a queda atinge todos os perfis de forma parecida, o problema é do serviço e não de um público específico
-- ATENÇÃO: assume gold.dim_cliente.regiao; ajustar o nome da coluna se for diferente
SELECT
  cl.faixa_idade,
  cl.regiao,
  COUNT_IF(d.ano = 2026)                                                           AS chamados_2026,
  ROUND(100 * COUNT_IF(d.ano = 2025 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2025 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(d.ano = 2026 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2026 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026
FROM gold.fato_chamado f
JOIN gold.dim_data    d  ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_cliente cl ON f.FK_cliente = cl.SK_cliente
WHERE d.SK_data <> -1 AND cl.SK_cliente <> -1
  AND cl.faixa_idade <> 'Não Informado'
  AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
GROUP BY cl.faixa_idade, cl.regiao
HAVING COUNT_IF(d.ano = 2026) > 30
ORDER BY (csat_2026 - csat_2025) ASC
LIMIT 20;

## 5. Atendentes: qualidade do app, do site e da URA

> "A alta demanda deve-se à qualidade do app e do site, que são lentos e possuem navegação difícil. Já ouvimos queixas dos usuários sobre a URA possuir caminhos confusos e de difícil navegação."

A base não tem dados de uso do app e do site, como tempo de carregamento, cliques ou erros, então a lentidão não pode ser medida diretamente. O relato é testado por sinais indiretos presentes nos chamados: o volume de problemas técnicos nos canais digitais, as demandas simples que mesmo assim chegam ao atendimento humano, o comportamento dos clientes depois de passar pela URA e o retorno de clientes pelo mesmo motivo em poucos dias.

### 5.1 Problemas técnicos nos canais digitais

Volume e participação mensal dos chamados da categoria Atendimento, que reúne problemas técnicos em site, app, chatbot e URA, com destaque para os problemas com o aplicativo.

In [0]:
%sql
-- Categoria 'Atendimento' = problemas técnicos em site, app, chatbot e URA (conforme dicionário do case)
SELECT
  d.ano,
  d.mes,
  COUNT(*)                                                           AS chamados,
  COUNT_IF(m.categoria = 'Atendimento')                              AS chamados_problema_tecnico,
  ROUND(100 * COUNT_IF(m.categoria = 'Atendimento') / COUNT(*), 1)   AS pct_problema_tecnico,
  COUNT_IF(m.nome LIKE '%plicativo%')                                AS chamados_problema_app
FROM gold.fato_chamado f
JOIN gold.dim_data   d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
WHERE d.SK_data <> -1 AND m.SK_motivo <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

### 5.2 Crescimento do volume por canal

Variação do número de chamados de cada canal entre janeiro e julho de 2025 e de 2026. Mostra se a demanda cresceu no autosserviço ou nos canais humanos.

In [0]:
%sql
-- A demanda cresceu onde? Em autosserviço (URA/Chatbot) ou nos canais humanos?
SELECT
  c.nome                                                   AS canal,
  c.tipo                                                   AS tipo_canal,
  COUNT_IF(d.ano = 2025)                                   AS chamados_2025,
  COUNT_IF(d.ano = 2026)                                   AS chamados_2026,
  ROUND(100 * (COUNT_IF(d.ano = 2026) - COUNT_IF(d.ano = 2025))
            / NULLIF(COUNT_IF(d.ano = 2025), 0), 1)        AS crescimento_pct
FROM gold.fato_chamado f
JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE d.SK_data <> -1 AND c.SK_canal <> -1
  AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
GROUP BY c.nome, c.tipo, c.nivel_etapa
ORDER BY c.nivel_etapa;

### 5.3 Funil real e funil desejado

Participação de cada canal no volume de cada semestre, com o custo médio. O funil desejado concentra o volume em prevenção e autosserviço.

In [0]:
%sql
-- O case quer menos volume nos níveis humanos e mais em Prevenção/Autosserviço.
-- Aqui medimos quanto do volume registrado está em autosserviço x humano, por semestre
SELECT
  d.ano,
  CASE WHEN d.mes <= 6 THEN '1º semestre' ELSE '2º semestre' END AS semestre,
  c.tipo                                                         AS tipo_canal,
  c.nome                                                         AS canal,
  COUNT(*)                                                       AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.ano, CASE WHEN d.mes <= 6 THEN '1º semestre' ELSE '2º semestre' END), 1) AS pct_do_semestre,
  ROUND(AVG(f.custo), 2)                                         AS custo_medio
FROM gold.fato_chamado f
JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE d.SK_data <> -1 AND c.SK_canal <> -1
GROUP BY d.ano, CASE WHEN d.mes <= 6 THEN '1º semestre' ELSE '2º semestre' END, c.tipo, c.nome, c.nivel_etapa
ORDER BY d.ano, 2, c.nivel_etapa;

### 5.4 Motivos simples que chegam ao atendimento humano

Consultas, dúvidas sobre pontos e alterações cadastrais deveriam ser resolvidas no app ou no site. A consulta mede quanto desses motivos chega ao atendimento humano e o custo envolvido.

In [0]:
%sql
-- Consultas de fatura, limite, contrato, pontos e alterações cadastrais deveriam resolver em Prevenção (site/app).
-- Cada uma que chega ao humano é sinal de que o app/site não está cumprindo o papel
SELECT
  m.nome                                                              AS motivo,
  m.categoria,
  COUNT(*)                                                            AS chamados_total,
  COUNT_IF(c.tipo = 'Humano')                                         AS chamados_humano,
  ROUND(100 * COUNT_IF(c.tipo = 'Humano') / COUNT(*), 1)              AS pct_no_humano,
  ROUND(SUM(CASE WHEN c.tipo = 'Humano' THEN f.custo ELSE 0 END), 2)  AS custo_humano,
  ROUND(AVG(CASE WHEN c.tipo = 'Humano' THEN f.custo END), 2)         AS custo_medio_humano,
  ROUND(AVG(CASE WHEN c.tipo <> 'Humano' THEN f.custo END), 2)        AS custo_medio_autosservico
FROM gold.fato_chamado f
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
JOIN gold.dim_canal  c ON f.FK_canal  = c.SK_canal
WHERE m.SK_motivo <> -1 AND c.SK_canal <> -1
  AND (m.nome LIKE 'Consulta%' OR m.nome LIKE '%ontos%' OR m.categoria = 'Cadastral')
GROUP BY m.nome, m.categoria
ORDER BY custo_humano DESC;

### 5.5 URA e chatbot: escape para o atendimento humano

Para os canais de autosserviço, mede resolução, abandono e o percentual de clientes que abrem um chamado em canal humano até um dia depois. Um escape alto indica navegação confusa.

In [0]:
%sql
-- Sinal de URA confusa: cliente desiste/não resolve na URA e abre novo chamado em até 1 dia em canal humano
-- ATENÇÃO: assume gold.dim_data.data (DATE). Granularidade diária: o "escape" mede até 1 dia depois
WITH seq AS (
  SELECT
    f.FK_cliente,
    c.nome AS canal,
    c.tipo AS tipo_canal,
    f.FK_status_chamado AS status,
    f.tempo_atendimento,
    d.data AS dt,
    LEAD(c.tipo) OVER (PARTITION BY f.FK_cliente ORDER BY d.data, c.nivel_etapa) AS prox_tipo,
    LEAD(d.data) OVER (PARTITION BY f.FK_cliente ORDER BY d.data, c.nivel_etapa) AS prox_dt
  FROM gold.fato_chamado f
  JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
  WHERE d.SK_data <> -1 AND c.SK_canal <> -1 AND f.FK_cliente <> -1
)
SELECT
  canal,
  COUNT(*)                                                           AS chamados,
  ROUND(100 * COUNT_IF(status = 1) / COUNT(*), 1)                    AS pct_resolucao,
  ROUND(100 * COUNT_IF(status = 3) / COUNT(*), 1)                    AS pct_abandono,
  ROUND(AVG(tempo_atendimento), 1)                                   AS tempo_medio_seg,
  ROUND(100 * COUNT_IF(prox_tipo = 'Humano' AND DATEDIFF(prox_dt, dt) <= 1) / COUNT(*), 1)
                                                                     AS pct_escape_para_humano_1d,
  ROUND(100 * COUNT_IF(status <> 1 AND prox_tipo = 'Humano' AND DATEDIFF(prox_dt, dt) <= 1)
            / NULLIF(COUNT_IF(status <> 1), 0), 1)                   AS pct_escape_quando_nao_resolveu
FROM seq
WHERE tipo_canal <> 'Humano'
GROUP BY canal
ORDER BY pct_escape_para_humano_1d DESC;

### 5.6 Recontato em até 7 dias

Percentual de chamados em que o cliente volta pelo mesmo motivo em até 7 dias, e o custo desse retrabalho, por canal.

In [0]:
%sql
-- Proxy de "a primeira tentativa não resolveu", seja por navegação confusa ou por atendimento ineficaz
-- ATENÇÃO: assume gold.dim_data.data (DATE)
WITH seq AS (
  SELECT
    c.nome AS canal,
    f.custo,
    d.data AS dt,
    LEAD(d.data)       OVER (PARTITION BY f.FK_cliente, f.FK_motivo ORDER BY d.data) AS prox_dt
  FROM gold.fato_chamado f
  JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
  WHERE d.SK_data <> -1 AND c.SK_canal <> -1
    AND f.FK_cliente <> -1 AND f.FK_motivo <> -1
)
SELECT
  canal,
  COUNT(*)                                                         AS chamados,
  COUNT_IF(DATEDIFF(prox_dt, dt) <= 7)                             AS chamados_com_recontato,
  ROUND(100 * COUNT_IF(DATEDIFF(prox_dt, dt) <= 7) / COUNT(*), 1)  AS pct_recontato_7d,
  ROUND(SUM(CASE WHEN DATEDIFF(prox_dt, dt) <= 7 THEN custo ELSE 0 END), 2) AS custo_retrabalho
FROM seq
GROUP BY canal
ORDER BY pct_recontato_7d DESC;

### 5.7 Estimativa de ganho

Estimativa da economia anual se parte dos motivos simples atendidos por humanos passar a ser resolvida no autosserviço. O cálculo multiplica o número desses chamados no atendimento humano pela taxa de migração e pela diferença de custo médio entre o atendimento humano e o autosserviço, e anualiza o resultado. A taxa de migração de 30% é uma premissa e deve ser validada com a V-Credit.

In [0]:
%sql
-- Proposta: melhorar app/site e URA para que parte dos motivos simples seja resolvida em autosserviço
-- Ganho anual estimado = chamados simples no humano (anualizado) x taxa de migração x (custo humano − custo autosserviço)
WITH params AS (
  SELECT 0.30 AS taxa_migracao          -- premissa: 30% desses chamados passam a resolver em autosserviço
),
simples AS (
  SELECT
    COUNT_IF(c.tipo = 'Humano')                      AS chamados_humano,
    AVG(CASE WHEN c.tipo = 'Humano'  THEN f.custo END) AS custo_medio_humano,
    AVG(CASE WHEN c.tipo <> 'Humano' THEN f.custo END) AS custo_medio_autosservico,
    COUNT(DISTINCT CONCAT(d.ano, '-', d.mes))        AS meses_na_base
  FROM gold.fato_chamado f
  JOIN gold.dim_data   d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
  JOIN gold.dim_canal  c ON f.FK_canal  = c.SK_canal
  WHERE d.SK_data <> -1 AND m.SK_motivo <> -1 AND c.SK_canal <> -1
    AND (m.nome LIKE 'Consulta%' OR m.nome LIKE '%ontos%' OR m.categoria = 'Cadastral')
)
SELECT
  s.chamados_humano,
  ROUND(s.custo_medio_humano, 2)       AS custo_medio_humano,
  ROUND(s.custo_medio_autosservico, 2) AS custo_medio_autosservico,
  p.taxa_migracao,
  ROUND(s.chamados_humano * p.taxa_migracao
        * (s.custo_medio_humano - s.custo_medio_autosservico), 2)                      AS economia_no_periodo_rs,
  ROUND(s.chamados_humano * p.taxa_migracao
        * (s.custo_medio_humano - s.custo_medio_autosservico) / s.meses_na_base * 12, 2) AS economia_anualizada_rs
FROM simples s CROSS JOIN params p;